# 01 — Feature Extraction

**SONATAM** — *Symbolic Ontology & Neural Audio Transcription Architecture for Music*

This notebook walks through the **dual-branch feature extraction** pipeline:

1. Discover MSD tracks + matched MIDI files (with DTW quality filtering)
2. Extract **statistical features** via jSymbolic2 (≈200 numeric features)
3. Extract **semantic features** via musif / music21 (Roman numerals, key profiles, functional harmony)
4. Merge into a single wide DataFrame ready for KG construction + GNN training

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import json
from pathlib import Path

import pandas as pd

from sonata.config.settings import CFG
from sonata.notebook_utils import rp, pp, show_paths  # pp() = absolute path from config string
from sonata.data import LakhMSDLinker, read_msd_metadata

print("Config loaded ✓")
print("Raw data paths:")
for key in ("raw_dir", "midi_root", "h5_root", "match_scores_path"):
    p = pp(CFG["data"][key])
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {rp(p)}")

Config loaded ✓


ValueError: too many values to unpack (expected 2)

## 1. Discover & filter tracks

In [ ]:
# Load match scores for quality filtering  (pp() resolves to project root)
scores_path = pp(CFG["data"].get("match_scores_path", "data/raw/match_scores.json"))
match_scores = {}
if scores_path.exists():
    with open(scores_path) as f:
        match_scores = json.load(f)
    print(f"Loaded match scores for {len(match_scores):,} tracks")
else:
    print(f"match_scores not found at {rp(scores_path)} — will skip DTW filter")

match_scores not found at notebooks/data/raw/match_scores.json — will skip DTW filter


In [ ]:
# Initialise the dual-branch linker with resolved paths
linker = LakhMSDLinker(
    midi_root         = pp(CFG["data"]["midi_root"]),
    h5_root           = pp(CFG["data"]["h5_root"]),
    match_scores_path = scores_path,          # already resolved above
)

tracks = linker.discover_tracks()
print(f"Discovered {len(tracks):,} tracks")

## 2. Build the dataset (jSymbolic + Semantic)

The `build_dataset()` method runs both extraction branches and
merges everything into a single wide DataFrame with columns:
- MSD metadata (`track_id`, `artist_name`, `primary_genre`, `year`, …)
- `jsym_*` — statistical features from jSymbolic2
- `sem_*` — semantic features from musif / music21

In [ ]:
df = linker.build_dataset(
    tracks  = tracks[:500],  # set tracks (no slice) for full dataset
    verbose = True,
)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Feature column overview
jsym_cols = [c for c in df.columns if c.startswith("jsym_")]
sem_cols  = [c for c in df.columns if c.startswith("sem_")]
meta_cols = [c for c in df.columns if not c.startswith(("jsym_", "sem_"))]

print(f"Metadata columns:  {len(meta_cols)}")
print(f"jSymbolic columns: {len(jsym_cols)}")
print(f"Semantic columns:  {len(sem_cols)}")
print(f"Total columns:     {len(df.columns)}")

## 3. Save the curated dataset

In [ ]:
out_dir = pp(CFG["dataset"]["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = pp(CFG["dataset"]["parquet_file"])
csv_path     = pp(CFG["dataset"]["csv_file"])

df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

print(f"✓ Saved → {rp(parquet_path)}")
print(f"✓ Saved → {rp(csv_path)}")

## 4. Quick EDA

In [ ]:
import matplotlib.pyplot as plt

if "primary_genre" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    df["primary_genre"].value_counts().head(20).plot.barh(ax=ax)
    ax.set_title("Top 20 genres")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()

if "year" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 3))
    df["year"].dropna().astype(int).hist(bins=50, ax=ax)
    ax.set_title("Year distribution")
    plt.tight_layout()
    plt.show()